In [ ]:
!git clone https://github.com/trangphan10/GNN-product-recommendation.git

In [ ]:
%cd  /kaggle/working/GNN-product-recommendation

In [ ]:
!pip install \
torch==2.5.1 \
torchvision==0.20.1 \
torchaudio==2.5.1 \
torch-geometric==2.6.1 \
torch-sparse==0.6.18 \
torch_scatter \
torchdata==0.9.0 \
dgl==2.1.0 \
ogb==1.3.6 \
lightning==2.5.5 \
torchmetrics==1.8.2 \
tensorboardX==2.6.4 \
tqdm==4.67.1 \
scipy==1.15.3 \
pandas \
matplotlib \
scikit-learn \
faiss-cpu==1.14.2 \
ipykernel\
jupyterlab \
-f https://data.pyg.org/whl/torch-2.5.1+cpu.html

In [ ]:
from torch_geometric.datasets import Amazon
Amazon(root='/kaggle/working/GNN-product-recommendation/data', name='Computers')
print("Done")

In [ ]:
%cd /kaggle/working/GNN-product-recommendation

In [ ]:
import os
os.chdir('/kaggle/working/GNN-product-recommendation')
os.makedirs('data', exist_ok=True)

from torch_geometric.datasets import Amazon
ds = Amazon(root='data', name='Computers')
print(f"Dataset ready: {ds}")


In [ ]:
# Patch load_dataset.py — thêm pattern tên mới của PyG 2.x
with open('load_dataset.py', 'r') as f:
    src = f.read()

old = '''def find_amazon_computer_npz(root):
    root = Path(root)
    candidates = sorted(root.glob("amazon_co_buy_computer*/amazon_co_buy_computer.npz"))
    candidates += sorted(root.glob("**/amazon_co_buy_computer.npz"))
    if not candidates:
        _download_amazon_computer(root)
        # Search again after download (PyG saves under root/Amazon/Computers/raw/)
        candidates = sorted(root.glob("amazon_co_buy_computer*/amazon_co_buy_computer.npz"))
        candidates += sorted(root.glob("**/amazon_co_buy_computer.npz"))
    if not candidates:
        raise FileNotFoundError(f"amazon_co_buy_computer.npz not found under {root}")
    return candidates[0]'''

new = '''def find_amazon_computer_npz(root):
    root = Path(root)
    candidates = sorted(root.glob("amazon_co_buy_computer*/amazon_co_buy_computer.npz"))
    candidates += sorted(root.glob("**/amazon_co_buy_computer.npz"))
    candidates += sorted(root.glob("**/amazon_electronics_computers.npz"))
    if not candidates:
        raise FileNotFoundError(f"Dataset NPZ not found under {root}")
    return candidates[0]'''

if old in src:
    with open('load_dataset.py', 'w') as f:
        f.write(src.replace(old, new))
    print("Patched OK")
else:
    print("Pattern not matched — paste src below:")
    print(src[src.find('def find_amazon'):src.find('def find_amazon')+400])


In [ ]:
!python train_resnext_gamlp.py --mode plain --cache-features

In [ ]:
!python train_improved_gamlp.py --mode rlu --label-smooth-eps 0.0 --label-decay 1.0  # tắt SDLP


In [ ]:
!python train_improved_gamlp.py --mode rlu --att-sparsity 0.0                         # tắt SHA  

In [ ]:
!python train_improved_gamlp.py --mode rlu --no-dynamic-threshold                     # tắt CAPS

## Baseline

In [ ]:
# MLP — không dùng đồ thị, baseline rẻ nhất
!python train_mlp.py

In [ ]:
# GCN
!python train_gcn.py

In [ ]:

# GraphSAGE — full-batch (mặc định, phù hợp vì đồ thị nhỏ)
!python train_graphsage.py

In [ ]:
# GraphSAGE — mini-batch neighbor sampling (bộ nhớ huấn luyện O(batch_size × fanout^layers),
# minh hoạ cách scale khi đồ thị lớn hơn nhiều so với RAM/VRAM)
!python train_graphsage.py --sampling --batch-size 512 --fanout 10 10

In [ ]:
# GAT — multi-head attention, heads/head-dim nhỏ theo mặc định
!python train_gat.py

In [ ]:
# Graph Transformer — attention chỉ tính trên các cạnh của đồ thị (O(E)), không phải
# attention toàn cục O(N²) như Transformer thường
!python train_graph_transformer.py

## Improved GraphSAGE — 3 cải tiến nhắm đúng điểm yếu

| # | Điểm yếu | Cải tiến | Flag |
|---|-----------|----------|------|
| 1 | Mean aggregator non-injective | Multi-aggregator {mean, max, std} | `--no-multi-aggr` để tắt |
| 2 | Over-smoothing khi nhiều lớp | Jumping Knowledge (JK-concat) | `--jk-mode none` để tắt |
| 3 | Uniform sampling bỏ sót node hiếm | Importance sampling ∝ 1/√degree | `--no-importance-sampling` để tắt |

In [ ]:
# Ablation — tắt từng cải tiến để đo đóng góp riêng lẻ
# Tắt multi-aggregator (về mean-only)
!python train_improved_graphsage.py --no-multi-aggr --jk-mode concat

# Tắt JK (về last-layer only)
!python train_improved_graphsage.py --jk-mode none

# Tắt cả hai (≈ GraphSAGE gốc nhưng 3 lớp)
!python train_improved_graphsage.py --no-multi-aggr --jk-mode none

In [ ]:
# Improved GraphSAGE — tất cả 3 cải tiến bật (full-batch, nhanh nhất để kiểm tra)
!python train_improved_graphsage.py --num-layers 3 --hidden 128 --jk-mode concat

In [ ]:

# ── So sánh kết quả tất cả mô hình ─────────────────────────────────────────
import json, glob, os
from pathlib import Path
from collections import defaultdict

# Tìm tất cả results.json trong outputs/
results_files = sorted(glob.glob("outputs/**/results.json", recursive=True))

# Gom theo tên model (lấy best_test cao nhất nếu có nhiều run)
rows = defaultdict(lambda: {"best_val": -1, "best_test": -1, "runs": 0})
MODEL_ORDER = [
    "mlp", "gcn", "graphsage", "graphsage_sampled",
    "gat", "graph_transformer",
    "gamlp_plain", "gamlp_rlu",
    "improved_gamlp_plain", "improved_gamlp_rlu",
    "resnext_gamlp_plain", "resnext_gamlp_rlu",
]

for path in results_files:
    with open(path) as f:
        r = json.load(f)

    # Tìm tên model từ field "model" hoặc suy từ đường dẫn
    model = r.get("model") or r.get("run_name", "")
    mode  = r.get("mode", "")
    if not model:
        parts = Path(path).parts
        model = next((p for p in parts if p in MODEL_ORDER), parts[-3])

    # Gắn mode vào key nếu có (plain / rlu)
    key = f"{model}_{mode}" if mode else model
    key = key.strip("_")

    val  = r.get("best_val",  r.get("valid_acc", -1)) or -1
    test = r.get("best_test", r.get("test_acc",  -1)) or -1
    rows[key]["best_val"]  = max(rows[key]["best_val"],  float(val))
    rows[key]["best_test"] = max(rows[key]["best_test"], float(test))
    rows[key]["runs"] += 1

if not rows:
    print("Chưa có kết quả nào trong outputs/. Hãy chạy ít nhất một training script trước.")
else:
    # Sắp xếp: model_order trước, rồi theo best_test giảm dần
    def sort_key(item):
        k = item[0]
        base = k.rsplit("_plain", 1)[0].rsplit("_rlu", 1)[0]
        idx = MODEL_ORDER.index(base) if base in MODEL_ORDER else 99
        return (idx, "plain" not in k)   # plain trước rlu

    sorted_rows = sorted(rows.items(), key=sort_key)

    # In bảng
    header = f"{'Model':<30} {'Val Acc':>8} {'Test Acc':>9} {'Runs':>5}"
    sep    = "-" * len(header)
    print(sep)
    print(header)
    print(sep)
    best_test_overall = max(v["best_test"] for v in rows.values())
    for key, v in sorted_rows:
        val_s  = f"{v['best_val']:.4f}"  if v['best_val']  >= 0 else "  —   "
        test_s = f"{v['best_test']:.4f}" if v['best_test'] >= 0 else "  —   "
        marker = " ◀ best" if v["best_test"] == best_test_overall else ""
        print(f"{key:<30} {val_s:>8} {test_s:>9} {v['runs']:>5}{marker}")
    print(sep)
    print(f"{'Split:':<30} split_idx.csv (8246 train / 2746 valid / 2760 test, cố định)")
